# DLMI 2026 Data Challenge: Histopathology OOD classification

**Team Name:** RATP
* **Alvaro Ruiz** - alvaro.ruiz@etu.u-paris.fr
* **Tom Pitois** - tom.pitois@protonmail.com

---

## Overview
This notebook contains our final solution for the binary classification of histopathology patches, specifically addressing the Out-Of-Distribution (OOD) / Covariate Shift problem across different medical centers.

Our pipeline integrates:
1. **Offline Macenko Normalization** to isolate color heuristics.
2. **Phikon (ViT-Base)**: A medical foundation model fine-tuned with differential learning rates.
3. **MixUp and CutMix**: Advanced batch-level data augmentation to prevent overfitting.
4. **Probability Calibration**: Dynamic threshold adjustment to compensate for the use of Soft Labels.

### Execution Warnings & Fast-Track
* **RAM Requirement (~30 GB):** To accelerate training by $2\times$, the computationally expensive Macenko normalization is applied once offline, and the entire processed dataset is loaded into RAM.
* **Fast-Track (Inference Only):** The end-to-end training takes approximately **6 hours** on a T4 GPU. If you wish to evaluate the model without waiting, **you can skip the Train/Val Dataset/DataLoader cells and the Fine-Tuning loop**. Proceed directly to the **"Load weights"** section to download our pre-trained optimal weights via Google Drive.

### Setup

#### Imports

In [1]:
# Install required libraries
!pip install torchstain transformers gdown

In [2]:
import os
import random
import warnings
import h5py
import torch
import torchstain
import copy
import gdown
import gc

import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import pandas as pd

from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm.notebook import tqdm
from transformers import ViTModel
from sklearn.metrics import roc_auc_score, accuracy_score

# Paths configuration
ROOT_DIR = Path('/kaggle/input/competitions/mva-dlmi-2026-histopathology-ood-classification/')
TRAIN_IMAGES_PATH = ROOT_DIR / 'train.h5'
VAL_IMAGES_PATH = ROOT_DIR / 'val.h5'
TEST_IMAGES_PATH = ROOT_DIR / 'test.h5'

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


#### Reproducibility

In [3]:
def seed_everything(seed=42):
    """
    Seeds all random number generators to ensure reproducibility.
    """
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # Configure CuDNN for reproducibility
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42
seed_everything(SEED)
print(f"Seeds have been set to {SEED}.")

Seeds have been set to 42.


## Macenko Normalizer

Fit the Macenko normalizer exclusively on Center 0 (our validation set) to prevent data leakage. The fitted normalizer is used by the Dataset to apply stain normalization when loading the data.

In [4]:
def get_macenko_normalizer(h5_path, target_center=0):
    """
    Finds a valid image from the target center and fits the Macenko Normalizer.
    """
    print(f"Looking for a reference image from Center {target_center}...")
    normalizer = torchstain.normalizers.MacenkoNormalizer(backend='numpy')
    
    with h5py.File(h5_path, 'r') as hdf:
        for img_idx in hdf.keys():
            item = hdf.get(img_idx)
            center = int(np.array(item.get('metadata'))[0])
            
            if center == target_center:
                img_np = np.array(item.get('img'))
                
                # Transpose to (H, W, C) for torchstain
                if img_np.shape[0] == 3:
                    img_np = np.transpose(img_np, (1, 2, 0))
                
                # Ensure uint8 format [0, 255]
                if img_np.max() <= 1.0:
                    img_np = (img_np * 255).astype(np.uint8)
                else:
                    img_np = img_np.astype(np.uint8)
                
                try:
                    normalizer.fit(img_np)
                    print(f"Success! Normalizer fitted using image ID: {img_idx}")
                    return normalizer
                except Exception:
                    # Fails if the image is mostly white background; try the next one
                    continue
                    
    raise ValueError(f"Could not fit Macenko normalizer for Center {target_center}.")

# Initialize the normalizer
macenko_norm = get_macenko_normalizer(TRAIN_IMAGES_PATH, target_center=0)

Looking for a reference image from Center 0...
Success! Normalizer fitted using image ID: 10004


## Dataset

Offline Normalization & RAM-Optimized Dataset

Calculating Macenko stain vectors "on-the-fly" within the PyTorch DataLoader creates a severe CPU bottleneck. To solve this, we made an engineering trade-off: we sacrifice RAM to maximize GPU utilization. This custom Dataset applies the Macenko normalizer once during initialization and stores the resulting float tensors in memory.

In [5]:
class NormalizedHistopathologyDataset(Dataset):
    """
    Dataset class that loads images, applies Macenko normalization offline, 
    and stores them in RAM for fast training.
    """
    def __init__(self, h5_path, normalizer, transform=None, is_test=False, centers_to_keep=None):
        self.transform = transform
        self.is_test = is_test
        self.images = []
        self.labels = []
        self.ids = []
        
        center_filter_txt = f"(Centers: {centers_to_keep})" if centers_to_keep is not None else "(All Centers)"
        print(f"Loading and Normalizing {h5_path.name} {center_filter_txt}...")
        
        with h5py.File(h5_path, 'r') as hdf:
            indices = list(hdf.keys())
            
            for img_idx in tqdm(indices, desc="Processing images"):
                item = hdf.get(img_idx)
                
                # Filter by center if we are not in the test set
                if not self.is_test:
                    center = int(np.array(item.get('metadata'))[0])
                    if centers_to_keep is not None and center not in centers_to_keep:
                        continue
                
                img_np = np.array(item.get('img'))
                
                # Prepare shape (H, W, C) and type uint8 for Macenko
                if img_np.shape[0] == 3:
                    img_np = np.transpose(img_np, (1, 2, 0))
                
                if img_np.max() <= 1.0:
                    img_np_uint8 = (img_np * 255).astype(np.uint8)
                else:
                    img_np_uint8 = img_np.astype(np.uint8)
                
                # Apply Macenko Normalization safely
                norm_img = img_np_uint8 # Fallback to original if normalization fails
                try:
                    with warnings.catch_warnings():
                        warnings.simplefilter("error", RuntimeWarning)
                        norm_img_tmp, _, _ = normalizer.normalize(I=img_np_uint8, stains=False)
                        
                        if not np.isnan(norm_img_tmp).any():
                            norm_img = norm_img_tmp
                except Exception:
                    # Fails silently for empty/background patches and keeps the original
                    pass
                
                # Convert back to PyTorch format (C, H, W) and float32 [0, 1]
                norm_img = np.transpose(norm_img, (2, 0, 1))
                img_tensor = torch.tensor(norm_img, dtype=torch.float32) / 255.0
                
                self.images.append(img_tensor)
                self.ids.append(img_idx)
                
                if not self.is_test:
                    self.labels.append(int(np.array(item.get('label'))))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        
        # Apply data augmentations (flips, rotations, resizes)
        if self.transform:
            img = self.transform(img)
            
        if self.is_test:
            return img, self.ids[idx]
            
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return img, label

## Transformations

We need to resize the images from their original $96\times 96$ shape to the $224 \times 224$ required by Phikon. We also apply some random transformations such as flips and rotations, leaving the issue of color entirely to Macenko's stain standardization.

In [6]:
# Standard spatial transforms for training
train_transforms = transforms.Compose([
    transforms.ToPILImage(), # Convert tensor from dataset back to PIL for resizing
    transforms.Resize((224, 224)), # Phikon expects 224x224
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(90),
    transforms.ToTensor(),
    # ImageNet stats
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
])

# Transforms for validation and test
val_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## Dataloaders

To ensure our validation metrics are unbiased and perfectly simulate the Kaggle blind evaluation, we avoid standard random splits (which cause data leakage of scanner characteristics). We isolate the centers as follows:
* **Train (Centers 3 & 4):** Used for training due to their high variance.
* **Validation (Center 0):** Strictly reserved for hyperparameter tuning and early stopping.
* **Internal Test (Center 1):** Acts as our internal proxy for the Kaggle Leaderboard to evaluate true OOD generalization.

> **FAST-TRACK NOTE:** If you are only performing inference, you **do not** need to execute the Dataset and DataLoader cells for Train and Validation. You can instantiate only the Test sets later.

In [7]:
# Train on Centers 3 and 4
train_dataset = NormalizedHistopathologyDataset(
    TRAIN_IMAGES_PATH, normalizer=macenko_norm, transform=train_transforms, centers_to_keep=[3, 4]
)

# Validation on Center 0
val_dataset = NormalizedHistopathologyDataset(
    TRAIN_IMAGES_PATH, normalizer=macenko_norm, transform=val_transforms, centers_to_keep=[0]
)

# Internal Strict OOD Test on Center 1
test_dataset = NormalizedHistopathologyDataset(
    VAL_IMAGES_PATH, normalizer=macenko_norm, transform=val_transforms
)

# Create DataLoaders
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Datasets loaded! Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

Loading and Normalizing train.h5 (Centers: [3, 4])...


Processing images:   0%|          | 0/100000 [00:00<?, ?it/s]

Loading and Normalizing train.h5 (Centers: [0])...


Processing images:   0%|          | 0/100000 [00:00<?, ?it/s]

Loading and Normalizing val.h5 (All Centers)...


Processing images:   0%|          | 0/34904 [00:00<?, ?it/s]

Datasets loaded! Train: 82244 | Val: 17756 | Test: 34904


## MixUp & CutMix

We implement a dynamic batch-level augmentation strategy. By fusing random pairs of images and assigning them "soft labels" (e.g., $0.3$ healthy, $0.7$ tumor), we force the network to focus on localized morphological structures rather than memorizing global artifacts.

In [8]:
def apply_mixup_cutmix(images, labels, alpha=0.2, cutmix_prob=0.5):
    """
    Applies either MixUp or CutMix to a batch of images and their labels.
    This acts as a strong regularizer for Out-Of-Distribution generalization.
    """
    batch_size = images.size(0)
    
    # Shuffle the batch to get pairs
    indices = torch.randperm(batch_size).to(images.device)
    shuffled_images = images[indices]
    shuffled_labels = labels[indices]
    
    # Decide between CutMix and MixUp based on probability
    if np.random.rand() < cutmix_prob:
        # CUTMIX
        lam = np.random.beta(alpha, alpha)
        
        # Calculate random bounding box
        H, W = images.size(2), images.size(3)
        cut_rat = np.sqrt(1. - lam)
        cut_w = int(W * cut_rat)
        cut_h = int(H * cut_rat)
        
        cx = np.random.randint(W)
        cy = np.random.randint(H)
        
        bbx1 = np.clip(cx - cut_w // 2, 0, W)
        bby1 = np.clip(cy - cut_h // 2, 0, H)
        bbx2 = np.clip(cx + cut_w // 2, 0, W)
        bby2 = np.clip(cy + cut_h // 2, 0, H)
        
        # Apply the patch swap
        images[:, :, bby1:bby2, bbx1:bbx2] = shuffled_images[:, :, bby1:bby2, bbx1:bbx2]
        
        # Adjust lambda to the exact pixel ratio swapped
        lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (images.size()[-1] * images.size()[-2]))
    else:
        # MIXUP
        lam = np.random.beta(alpha, alpha)
        images = lam * images + (1 - lam) * shuffled_images
        
    # Create the new soft labels
    mixed_labels = lam * labels + (1 - lam) * shuffled_labels
    return images, mixed_labels

## Model

We replace standard ImageNet models with **Phikon (ViT-Base)**, pre-trained on the TCGA pan-cancer dataset. To prevent catastrophic forgetting of its medical priors, we freeze the initial layers and only unfreeze the last 4 transformer encoder blocks alongside a custom non-linear classification head.

> **FAST-TRACK NOTE:** Even if you are only performing inference, you **need** to execute this cell in order to instantiate the model.

In [9]:
class FineTunedMedicalModel(nn.Module):
    def __init__(self, num_unfrozen_blocks=4):
        """
        Loads Phikon and unfreezes only the last N blocks and the classification head.
        This prevents destroying the pre-trained medical knowledge while allowing adaptation.
        """
        super(FineTunedMedicalModel, self).__init__()
        
        print("Downloading/Loading Phikon from HuggingFace...")
        self.backbone = ViTModel.from_pretrained("owkin/phikon", add_pooling_layer=False)
        
        # Freeze EVERYTHING first
        for param in self.backbone.parameters():
            param.requires_grad = False
            
        # Unfreeze the last N transformer encoder blocks
        # Phikon (ViT-Base) has 12 blocks (0 to 11). We unfreeze the last ones.
        if num_unfrozen_blocks > 0:
            print(f"Unfreezing the last {num_unfrozen_blocks} transformer blocks for Fine-Tuning...")
            encoder_blocks = self.backbone.encoder.layer
            total_blocks = len(encoder_blocks)
            
            for i in range(total_blocks - num_unfrozen_blocks, total_blocks):
                for param in encoder_blocks[i].parameters():
                    param.requires_grad = True
                    
            # Also unfreeze the final LayerNorm of the backbone
            for param in self.backbone.layernorm.parameters():
                param.requires_grad = True

        # Create our classification head (Binary output)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(768, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        outputs = self.backbone(x)
        cls_token = outputs.last_hidden_state[:, 0, :] # Extract [CLS] token
        return self.classifier(cls_token)

# Instantiate the model while unfreezing the last 4 blocks
model = FineTunedMedicalModel(num_unfrozen_blocks=4).to(device)

Downloading/Loading Phikon from HuggingFace...


config.json:   0%|          | 0.00/492 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: owkin/phikon
Key                 | Status     |  | 
--------------------+------------+--+-
pooler.dense.weight | UNEXPECTED |  | 
pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Unfreezing the last 4 transformer blocks for Fine-Tuning...


In [10]:
model

FineTunedMedicalModel(
  (backbone): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (interme

## Optimization

We apply end-to-end fine-tuning using a differential learning rate (slower for the backbone, faster for the head) and an automatic ReduceLROnPlateau scheduler. The loss function is Binary CE with logits.

> **FAST-TRACK NOTE:** Even if you are only performing inference, you **need** to execute this cell.

In [11]:
# Loss function for binary classification
# BCEWithLogitsLoss automatically handles the Sigmoid activation and works well with float labels from MixUp
criterion = nn.BCEWithLogitsLoss()

# Optimizer: AdamW
# We use a relatively small Learning Rate to avoid catastrophic forgetting of the pretrained weights
optimizer = torch.optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': 5e-5},
    {'params': model.classifier.parameters(), 'lr': 1e-3}
], weight_decay=1e-4)

# Learning Rate Scheduler: Reduces LR when the validation AUC stops improving
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

epochs = 6
best_val_auc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())
WEIGHTS_PATH = 'best_finetuned_phikon.pth'

## Fine-tuning

> **TIME WARNING:** Executing this cell takes approximately **4 hours**.

> **FAST-TRACK NOTE:** If you are only performing inference, you **do not** need to execute this cell. To skip this step, ignore this cell and proceed directly to the next section (**"Load weights"**).

In [12]:
print("Starting Fine-Tuning with MixUp/CutMix...")
print(f"Saving best model weights in {WEIGHTS_PATH}")

for epoch in range(epochs):
    print(f"\n[+] Epoch {epoch+1}/{epochs}")
    
    # ==========================================
    #                  TRAINING
    # ==========================================
    model.train()
    train_loss = 0.0
    
    for images, labels in tqdm(train_loader, desc="Training (Centers 3 & 4)"):
        images = images.to(device)
        labels = labels.to(device)
        
        # Apply CutMix or MixUp for extreme regularization
        # We apply it to 30% of the batches (cutmix_prob parameter handles the internal probability)
        # Note: We need to make sure labels are strictly floats for BCEWithLogitsLoss
        labels = labels.float()
        images, mixed_labels = apply_mixup_cutmix(images, labels, alpha=0.2, cutmix_prob=0.3)
        
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(images).squeeze(1)
        
        # Calculate loss with the mixed soft-labels
        loss = criterion(outputs, mixed_labels)
        
        # Backward pass and optimize
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        
    epoch_train_loss = train_loss / len(train_dataset)
    
    # ==========================================
    #                 VALIDATION
    # ==========================================
    model.eval()
    val_loss = 0.0
    val_preds = []
    val_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Validation (Center 0)"):
            images = images.to(device)
            labels = labels.float().to(device)
            
            outputs = model(images).squeeze(1)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            
            # Apply Sigmoid to get probabilities in range [0, 1]
            probs = torch.sigmoid(outputs)
            
            val_preds.extend(probs.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
            
    epoch_val_loss = val_loss / len(val_dataset)
    
    # Calculate Metrics
    val_auc = roc_auc_score(val_labels, val_preds)
    val_acc = accuracy_score(val_labels, (np.array(val_preds) > 0.5).astype(int))
    
    print(f"Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")
    print(f"Val AUC: {val_auc:.4f} | Val Acc: {val_acc:.4f}")
    
    # Step the scheduler based on Validation AUC
    scheduler.step(val_auc)
    
    # Save the model if it's the best one so far
    if val_auc > best_val_auc:
        print(f" > New best model. Val AUC improved from {best_val_auc:.4f} to {val_auc:.4f}")
        best_val_auc = val_auc
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(best_model_wts, WEIGHTS_PATH)

print("\nTraining completed.")

Starting Fine-Tuning with MixUp/CutMix...
Saving best model weights in best_finetuned_phikon.pth

[+] Epoch 1/6


Training (Centers 3 & 4):   0%|          | 0/2571 [00:00<?, ?it/s]

Validation (Center 0):   0%|          | 0/555 [00:00<?, ?it/s]

Train Loss: 0.2142 | Val Loss: 0.1580
Val AUC: 0.9951 | Val Acc: 0.9607
 > New best model. Val AUC improved from 0.0000 to 0.9951

[+] Epoch 2/6


Training (Centers 3 & 4):   0%|          | 0/2571 [00:00<?, ?it/s]

...

Validation (Center 0):   0%|          | 0/555 [00:00<?, ?it/s]

Train Loss: 0.1710 | Val Loss: 0.0911
Val AUC: 0.9955 | Val Acc: 0.9753
 > New best model. Val AUC improved from 0.9951 to 0.9955

[+] Epoch 6/6


Training (Centers 3 & 4):   0%|          | 0/2571 [00:00<?, ?it/s]

Validation (Center 0):   0%|          | 0/555 [00:00<?, ?it/s]

Train Loss: 0.1691 | Val Loss: 0.0806
Val AUC: 0.9946 | Val Acc: 0.9775

Training completed.


### Load weights

If you opted for the fast-track and skipped the 4-hour training, this cell will automatically download our best model weights from Google Drive using `gdown`. If the model was trained locally in the previous step, it will simply load the generated `.pth` file.

In [13]:
if not os.path.exists(WEIGHTS_PATH):
    FILE_ID = '1LOW_qBMeOknZ4FSI7gLT3ubyZQ3kzdpM'
    URL = f'https://drive.google.com/uc?id={FILE_ID}'

    print("Downloading from Google Drive...")
    # Download from GDrive via gdown
    gdown.download(URL, WEIGHTS_PATH, quiet=False) 
    print("Download complete!")
    
else:
    print("Weights already found locally. Model has been trained. Skipping download.")

Weights already found locally. Model has been trained. Skipping download.


Once the weights have been downloaded if the model was not fine-tuned, they can be loaded.

In [14]:
# Load the best weights found during validation
print(f"Loading best model from {WEIGHTS_PATH}")
model.load_state_dict(torch.load(WEIGHTS_PATH))
model.to(device)
model.eval()

Loading best model from best_finetuned_phikon.pth


FineTunedMedicalModel(
  (backbone): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (interme

## Internal Test

Evaluate the model on the Center 1 dataset, which serves as our _Internal Test set_ (partition `val.h5`) to simulate the OOD effect that would occur with another dataset, just as it would on Kaggle. Estimated duration: **6 minutes**.

In [15]:
test_preds = []
test_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing (Center 1)"):
        images = images.to(device)
        
        outputs = model(images).squeeze(1)
        probs = torch.sigmoid(outputs)
        
        test_preds.extend(probs.cpu().numpy())
        test_labels.extend(labels.numpy())

# Calculate Final Metrics
test_auc = roc_auc_score(test_labels, test_preds)
test_acc = accuracy_score(test_labels, (np.array(test_preds) > 0.5).astype(int))

print(f"\n[+] FINAL RESULTS (Phikon + Macenko + MixUp/CutMix)")
print(f"\tTest (Center 1) AUC: {test_auc:.4f}")
print(f"\tTest (Center 1) Acc: {test_acc:.4f}")

Testing (Center 1):   0%|          | 0/1091 [00:00<?, ?it/s]


[+] FINAL RESULTS (Phikon + Macenko + MixUp/CutMix)
	Test (Center 1) AUC: 0.9850
	Test (Center 1) Acc: 0.9146


## Clearing

Remove some variables that will no longer be used.

In [18]:
del train_dataset, val_dataset, test_dataset
del train_loader, val_loader, test_loader
gc.collect()

117398

## Test Prediction (Kaggle Submission)

The prediction is done in this step. Therefore, a new dataset and dataloader containing the data from the actual test partition (`test.h5`) are required. The estimated execution time for this cell is **20 minutes**.

In [19]:
kaggle_test_dataset = NormalizedHistopathologyDataset(
    TEST_IMAGES_PATH, 
    normalizer=macenko_norm, 
    transform=val_transforms, 
    is_test=True 
)

kaggle_test_loader = DataLoader(
    kaggle_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)

submission_ids = []
submission_preds = []

with torch.no_grad():
    for images, img_ids in tqdm(kaggle_test_loader, desc="Inferring Real Test (test.h5)"):
        images = images.to(device)
        
        probs = torch.sigmoid(model(images).squeeze(1))        
        preds = (probs > 0.5).int().cpu().numpy()
        
        submission_ids.extend(img_ids.numpy() if isinstance(img_ids, torch.Tensor) else img_ids)
        submission_preds.extend(preds)

# Create DataFrame and CSV
submission_df = pd.DataFrame({
    'ID': submission_ids,
    'Pred': submission_preds
})

submission_df.to_csv('submission.csv', index=False)

Loading and Normalizing test.h5 (All Centers)...


Processing images:   0%|          | 0/85054 [00:00<?, ?it/s]

Inferring Real Test (test.h5):   0%|          | 0/2658 [00:00<?, ?it/s]

## Calibration

During our experiments, we observed that while the AUC was near-perfect ($>0.98$) on OOD data, the raw Accuracy dropped when using the standard `0.5` threshold. 

This is a direct mathematical consequence of our **MixUp/CutMix** regularization. By training the network with *Soft Labels*, the model learned to avoid extreme confidence, compressing its output probabilities towards the lower end of the distribution. For this highly regularized network, a prediction of `0.15` already means "tumor features detected with conservative confidence."

Here, we exclusively use our Validation Set (Center 0) to empirically find the optimal threshold, and then apply it to the unseen Internal Test (Center 1) to prove that the calibration generalizes across distributions.

In [21]:
# Find the best threshold using ONLY the Validation Set (Center 0)
best_threshold = 0.5
best_val_acc = 0.0

# Scan thresholds from 0.10 to 0.90 in steps of 0.01
thresholds = np.arange(0.1, 0.9, 0.01)

for thresh in tqdm(thresholds):
    # Convert probabilities to 0 or 1 based on the current threshold
    current_val_preds_bin = (np.array(val_preds) > thresh).astype(int)
    current_acc = accuracy_score(val_labels, current_val_preds_bin)
    
    if current_acc > best_val_acc:
        best_val_acc = current_acc
        best_threshold = thresh

print(f"Optimal Threshold found on Validation: {best_threshold:.2f} (Val Acc: {best_val_acc:.4f})")

# Apply this specific calibrated threshold to the Internal Test Set (Center 1)
test_preds_bin_calibrated = (np.array(test_preds) > best_threshold).astype(int)
calibrated_test_acc = accuracy_score(test_labels, test_preds_bin_calibrated)

print(f"\n[+] RE-CALIBRATED RESULTS (Center 1)")
print(f"\tOriginal Test Acc (0.50): {test_acc:.4f}")
print(f"\tCalibrated Test Acc ({best_threshold:.2f}): {calibrated_test_acc:.4f}")

  0%|          | 0/80 [00:00<?, ?it/s]

Optimal Threshold found on Validation: 0.11 (Val Acc: 0.9830)

[+] RE-CALIBRATED RESULTS (Center 1)
	Original Test Acc (0.50): 0.9146
	Calibrated Test Acc (0.11): 0.9499


## Test Prediction with Optimal Threshold (Kaggle Submission)

The final prediction for submission with optimal thresholding is done in this step. The estimated execution time for this cell is **15 minutes**.

In [22]:
submission_ids = []
submission_preds = []

with torch.no_grad():
    for images, img_ids in tqdm(kaggle_test_loader, desc="Inferring Real Test (test.h5)"):
        images = images.to(device)
        
        probs = torch.sigmoid(model(images).squeeze(1))        
        preds = (probs > best_threshold).int().cpu().numpy()
        
        submission_ids.extend(img_ids.numpy() if isinstance(img_ids, torch.Tensor) else img_ids)
        submission_preds.extend(preds)

# Create DataFrame and CSV
submission_df = pd.DataFrame({
    'ID': submission_ids,
    'Pred': submission_preds
})

submission_df.to_csv('submission_thresholded.csv', index=False)

Inferring Real Test (test.h5):   0%|          | 0/2658 [00:00<?, ?it/s]